# Smart Meter Data

This notebook downloads, processes, and samples smart meter consumption data consumption data from UK Power Networks.

## Importing Dependencies

In [1]:
import os, glob, time, shutil
from pathlib import Path
import numpy as np
import pandas as pd
from google.colab import drive
from shutil import copy2
import matplotlib.pyplot as plt
drive.mount("/content/drive")

Mounted at /content/drive


## Importing and Processing the Data

The smart meter data set from UK Power Networks (https://data.london.gov.uk/dataset/smartmeter-energy-use-data-in-london-households/) is vast - around 167 million rows. The data is partitioned into 168 csv files. This code downloads the files one by one, merges, and reformats the data into a global dataframe. For memory and storage purposes, the sample size is reduced by only keeping observations which correspond to weekdays and 'bridge months'.


In [2]:
folder = "/content/drive/MyDrive/ERP/Raw_Data/smart_meter_data"
random_state = 42


files = sorted(glob.glob(os.path.join(folder, "*.csv")))
print(f"Found {len(files)} CSVs")

def read_csv_with_retries(src_path, retries=3, sleep_s=2, use_local_copy=True):
    """Read a CSV from Drive with retries; optionally copy to /content first to avoid I/O hiccups."""
    p = Path(src_path)
    local_path = Path("/content") / p.name if use_local_copy else p

    for attempt in range(1, retries + 1):
        try:
            if use_local_copy:
                copy2(p, local_path)
            df = pd.read_csv(
                local_path,
                parse_dates=["DateTime"],
                dayfirst=False,
                low_memory=False
            )
            return df
        except Exception:
            if attempt == retries:
                raise
            time.sleep(sleep_s * attempt)
        finally:
            if use_local_copy and local_path.exists():
                try:
                    local_path.unlink()
                except Exception:
                    pass

all_data = []
for csv_path in files:
    name = Path(csv_path).stem
    print(f"Processing {name} ...")
    try:
        df = read_csv_with_retries(csv_path)


        df = df.drop_duplicates()
        df["DateTime"] = pd.to_datetime(df["DateTime"], errors="coerce")
        df = df.dropna(subset=["DateTime"])
        df["weekday"] = df["DateTime"].dt.weekday
        df = df[df["weekday"].between(0, 4)]
        tou_ids = set(df.loc[df["stdorToU"].astype(str).str.strip().eq("ToU"), "LCLid"].unique())
        if tou_ids:
            df = df[~df["LCLid"].isin(tou_ids)]
        kwh_col = "KWH/hh (per half hour) "
        df = df[["DateTime", "LCLid", kwh_col]].rename(columns={kwh_col: "kWh"})
        df["kWh"] = pd.to_numeric(df["kWh"], errors="coerce")
        all_data.append(df)
        print(f"  -> kept {df.shape[0]:,} rows")

    except Exception as e:
        print(f"  !! Failed {name}: {e}")

if not all_data:
    raise RuntimeError("No data loaded after filtering. Check inputs/filters.")

all_data = pd.concat(all_data, ignore_index=True)
print(f"\nCombined long table shape: {all_data.shape}")


wide = (
    all_data
    .pivot_table(index="DateTime", columns="LCLid", values="kWh", aggfunc="first")
    .sort_index()
    .sort_index(axis=1)
)
print(f"Wide shape: {wide.shape}")


hourly_sum   = wide.resample("1h").sum(min_count=2)
hourly_count = wide.resample("1h").count()

consumption_data = hourly_sum.where(hourly_count.ge(2))
print(f"Consumption shape after hourly filter: {consumption_data.shape}")


consumption_data = consumption_data[consumption_data.index.month.isin([3, 4, 9, 10])]
print(f"Consumption shape after month filter: {consumption_data.shape}")

consumption_data["date"] = consumption_data.index.normalize()
consumption_data["hour"] = consumption_data.index.hour

profiles = (
    consumption_data
    .melt(id_vars=["date", "hour"], var_name="household", value_name="kWh")
    .pivot_table(index=["date", "household"], columns="hour", values="kWh", aggfunc="first")
    .sort_index()
)

profiles = profiles.reindex(columns=sorted(profiles.columns))
profiles = profiles.apply(pd.to_numeric, errors="coerce")

hour_medians = profiles.median(axis=0, skipna=True)
profiles = profiles.fillna(hour_medians)

print("Profiles ready for clustering:")
display(profiles.head())
print(f"Final profiles shape: {profiles.shape}")


Found 168 CSVs
Processing LCL-June2015v2_0 ...
  -> kept 714,431 rows
Processing LCL-June2015v2_1 ...
  -> kept 713,718 rows
Processing LCL-June2015v2_10 ...
  -> kept 714,050 rows
Processing LCL-June2015v2_100 ...
  -> kept 714,105 rows
Processing LCL-June2015v2_101 ...
  -> kept 714,085 rows
Processing LCL-June2015v2_102 ...
  -> kept 713,730 rows
Processing LCL-June2015v2_103 ...
  -> kept 714,582 rows
Processing LCL-June2015v2_104 ...
  -> kept 714,784 rows
Processing LCL-June2015v2_105 ...
  -> kept 714,322 rows
Processing LCL-June2015v2_106 ...
  -> kept 714,834 rows
Processing LCL-June2015v2_107 ...
  -> kept 714,464 rows
Processing LCL-June2015v2_108 ...
  -> kept 713,914 rows
Processing LCL-June2015v2_109 ...
  -> kept 714,453 rows
Processing LCL-June2015v2_11 ...
  -> kept 714,940 rows
Processing LCL-June2015v2_110 ...
  -> kept 714,035 rows
Processing LCL-June2015v2_111 ...
  -> kept 713,595 rows
Processing LCL-June2015v2_112 ...
  -> kept 715,193 rows
Processing LCL-June201

hour                     0      1      2      3      4      5      6      7   \
date       household                                                           
2012-03-01 MAC000003  0.233  0.216  0.266  0.198  0.198  0.310  0.274  0.537   
           MAC000006  0.036  0.069  0.068  0.029  0.069  0.068  0.029  0.125   
           MAC000016  0.064  0.036  0.055  0.029  0.055  0.041  0.043  0.054   
           MAC000018  0.254  0.249  0.248  0.269  0.256  0.253  0.245  0.288   
           MAC000019  0.150  0.096  0.107  0.145  0.127  0.083  0.129  0.209   

hour                     8      9   ...     14     15     16     17     18  \
date       household                ...                                      
2012-03-01 MAC000003  0.603  0.500  ...  0.370  0.297  0.374  0.280  0.210   
           MAC000006  0.161  0.264  ...  0.170  0.167  0.098  0.095  0.216   
           MAC000016  0.030  0.054  ...  0.029  0.055  0.052  0.050  0.066   
           MAC000018  0.444  0.494  ...  0.524  0.589  0.414  0.664  0.652   
           MAC000019  0.155  0.411  ...  0.234  0.213  0.224  0.966  0.472   

hour                     19     20     21     22     23  
date       household                                     
2012-03-01 MAC000003  0.256  1.238  0.968  0.388  0.264  
           MAC000006  0.161  0.100  0.229  0.118  0.116  
           MAC000016  0.041  0.066  0.036  0.063  0.045  
           MAC000018  0.924  0.859  0.685  0.547  0.485  
           MAC000019  0.249  0.298  0.276  0.313  0.263  

[5 rows x 24 columns]

Final profiles shape: (589053, 24)


## Sampling

1000 random rows are sampled ensuring each household only features once:

In [3]:
n = 1000
seed = 42

one_per_household = (
    profiles
    .groupby(level="household", group_keys=False)
    .sample(1, random_state=seed)
)

if len(one_per_household) < n:
    raise ValueError(f"Requested {n} but only {len(one_per_household)} unique households available.")

demand_sample = one_per_household.sample(n, random_state=seed)


All observations are aggregated then scaled down to form an representative basleine demand profile:

In [4]:
hour_cols = [c for c in profiles.columns
             if isinstance(c, (int, np.integer)) or (isinstance(c, str) and c.isdigit())]
hour_cols = sorted(hour_cols, key=lambda x: int(x))

agg_by_hour = demand_sample[hour_cols].sum(axis=0)

avg_daily_demand = demand_sample.sum(axis=1).mean()
total = float(agg_by_hour.sum())
scale = avg_daily_demand / total
baseline_profile = agg_by_hour * scale
baseline_profile.sum()

np.float64(9.6438919982)

## Exporting data

In [5]:
baseline_profile.to_csv("/content/drive/MyDrive/ERP/Processed_Data/baseline_profile.csv")